# vps
> VPS provisioning: cloud-init generation, Hetzner hcloud CLI wrapper, SSH deployment helpers

In [ ]:
#| default_exp vps

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, json, subprocess, time
from fastcore.all import L, Path, run, listify, timed_cache
from fastcloudinit.core import cloud_init_base, cloud_init_config, user, runcmd, reboot
from fastops.core import Cli, mk_flags
from hcloud import Client
from hcloud.images import Image
from hcloud.server_types import ServerType
from hcloud.locations import Location
from hcloud.ssh_keys import SSHKey

## Multipass for local testing
> Using `multipass` to test cloud-init and deployment locally before provisioning real VPSes. You'll need to install Multipass and have it in your PATH for this to work.

In [ ]:
mk_flags(image='24.04', cpus=1, memory='1G', disk='10G', sym= ' ')

['--image 24.04', '--memory 1G', '--disk 10G']

In [ ]:
#| export
class Multipass(Cli):
    'Wrap multipass CLI: manage local Ubuntu VMs for testing'
    def _run(self, cmd, *args): return run('multipass', cmd, *args)
    def launch(self, name, image='24.04', cpus=1, memory='1G', disk='10G', cloud_init:Path=None, mounts=None):
        'Launch a VM. cloud_init can be a YAML string or a file path.'
        args = mk_flags(image=image, cpus=cpus, memory=memory, disk=disk, sym=' ')
        if cloud_init and Path(cloud_init).is_file(): args += mk_flags(cloud_init=str(cloud_init), sym=' ')
        for hp, vp in (mounts or {}).items(): args += mk_flags(mount=f'{hp}:{vp}', sym=' ')
        self('launch', name, *args)
        return name

    def vms(self, running=False):
        'List VM names. running=True filters to Running state.'
        lst = L(json.loads(self('list', '--format', 'json')).get('list', []))
        if running: lst = lst.filter(lambda v: v.get('state') == 'Running')
        return list(lst.itemgot('name'))

    def ip(self, name) -> str:
        'Get IPv4 address of a VM.'
        return json.loads(self('info', name, '--format', 'json'))['info'][name]['ipv4'][0]

    def exec_(self, name, *cmd) -> str:
        'Run a command in a VM.'
        return self('exec', name, '--', *cmd)

    def rm(self, name, purge=True) -> None:
        'Delete a VM.'
        self('delete', name)
        if purge: self('purge')

    def transfer(self, src, dst) -> None:
        'Transfer files to/from a VM. Use "vmname:/path" for VM paths.'
        self('transfer', src, dst)

mp = Multipass()

def deploy_mp(name, src, path='/srv/app', build=True) -> str:
    'Sync local directory into a Multipass VM and run docker compose up -d.'
    mp.exec_(name, 'mkdir', '-p', path)
    mp.transfer(str(src).rstrip('/') + '/', f'{name}:{path}')
    cmd = ['docker', 'compose', '-f', f'{path}/docker-compose.yml', 'up', '-d', '--remove-orphans']
    if build: cmd.append('--build')
    return mp.exec_(name, *cmd)

In [ ]:
mp.vms()

[]

## Cloud-init generation

`vps_init()` builds a cloud-init YAML for a fresh VPS via `fastcloudinit.cloud_init_config`: UFW hardening, user creation, SSH key setup, optional Docker. `multi_init()` is the local equivalent via `cloud_init_base` — same Docker setup, no UFW, no fail2ban.

In [ ]:
#| export
dock_cmd = [
    'curl -fsSL https://get.docker.com | sh',
    'usermod -aG docker {username}',
    'systemctl enable --now docker',
]

def vps_init(hostname, pub_keys, username='deploy', docker=True, pkgs=None, cmds=None, **kw):
    'Cloud-init YAML for a fresh VPS: user, UFW, optional Docker'
    rcmds = list(cmds or [])
    if docker: rcmds = [c.format(username=username) for c in dock_cmd] + rcmds
    default_pkgs = ['curl', 'fail2ban', 'unattended-upgrades']
    return cloud_init_config(hostname=hostname, username=username, pub_keys=pub_keys,
        packages=default_pkgs + list(pkgs or []), cmds=rcmds, **kw)

In [ ]:
#| export
def multi_init(hostname, pub_keys, username='deploy', docker=True, pkgs=None, cmds=None):
    'Cloud-init for Multipass local VMs: no UFW, no hardening, optional Docker'
    rcmds = list(cmds or [])
    if docker: rcmds = [c.format(username=username) for c in dock_cmd] + rcmds
    return cloud_init_base(
        hostname,
        packages=['curl'] + list(pkgs or []),
        users=[user(username, pub_keys)],
        **runcmd(rcmds),
        **reboot(),
    )

In [ ]:
yaml = multi_init('mylocal', 'ssh-rsa AAAA...')
assert '#cloud-config' in yaml
assert 'get.docker.com' in yaml
assert 'ufw' not in yaml
assert 'fail2ban' not in yaml
print(yaml)

#cloud-config
hostname: mylocal
preserve_hostname: false
packages:
- curl
package_update: true
package_upgrade: true
disable_root: true
ssh_pwauth: false
users:
- name: deploy
  groups:
  - sudo
  shell: /bin/bash
  sudo:
  - ALL=(ALL) NOPASSWD:ALL
  ssh_authorized_keys:
  - ssh-rsa AAAA...
runcmd:
- curl -fsSL https://get.docker.com | sh
- usermod -aG docker deploy
- systemctl enable --now docker
power_state:
  mode: reboot
  message: Rebooting
  timeout: 1
  condition: true



In [ ]:
yaml = vps_init('myserver', 'ssh-rsa AAAA...', docker=True)
assert '#cloud-config' in yaml
assert 'get.docker.com' in yaml
assert 'deploy' in yaml
assert 'fail2ban' in yaml
assert 'unattended-upgrades' in yaml
print(yaml)

#cloud-config
hostname: myserver
preserve_hostname: false
packages:
- curl
- fail2ban
- unattended-upgrades
package_update: true
package_upgrade: true
disable_root: true
ssh_pwauth: false
users:
- name: deploy
  groups:
  - sudo
  shell: /bin/bash
  sudo:
  - ALL=(ALL) NOPASSWD:ALL
  ssh_authorized_keys:
  - ssh-rsa AAAA...
runcmd:
- curl -fsSL https://get.docker.com | sh
- usermod -aG docker deploy
- systemctl enable --now docker
- ufw default deny incoming
- ufw default allow outgoing
- ufw logging off
- ufw allow 22/tcp
- ufw --force enable
apt:
  conf: 'APT::Periodic::Update-Package-Lists "1";

    APT::Periodic::Download-Upgradeable-Packages "1";

    APT::Periodic::AutocleanInterval "7";

    APT::Periodic::Unattended-Upgrade "0";

    Unattended-Upgrade::Automatic-Reboot "false";

    '
write_files:
- path: /etc/logrotate.d/00-cloud-init-global
  owner: root:root
  permissions: '0644'
  content: "/var/log/*.log {\n    weekly\n    rotate 7\n    compress\n    su root adm\n    crea

## Hetzner (hcloud Python SDK)

`_client()` builds a `Client` from `HCLOUD_TOKEN` in the environment. All provisioning functions use it — no CLI binary required, no config files to write.

In [ ]:
#| export
@timed_cache(300)
def hetz():
    token = os.environ.get('HCLOUD_TOKEN')
    if not token: raise ValueError('HCLOUD_TOKEN environment variable not set')
    return Client(token=token)

In [ ]:
#| export
def create(name, image='ubuntu-24.04', server_type='cx23', location=None, cloud_init=None, ssh_keys=None):
    'Create a Hetzner server. Returns IP.'
    resp = hetz().servers.create(
        name=name,
        server_type=ServerType(name=server_type),
        image=Image(name=image),
        location=Location(name=location) if location else None,
        user_data=cloud_init,
        ssh_keys=[SSHKey(name=k) for k in (ssh_keys or [])],
    )
    return {resp.server.public_net.ipv4.ip:resp}

def servers():
    'List Hetzner servers as [{name, ip, status}]'
    return L(hetz().servers.get_all()).map(lambda s: dict(name=s.name,ip=s.public_net.ipv4.ip,status=s.status))

def server_ip(name) -> str:
    'Get public IPv4 of a Hetzner server by name'
    s = hetz().servers.get_by_name(name)
    if not s: raise ValueError(f'Server {name!r} not found')
    return s.public_net.ipv4.ip

def delete(name):
    'Delete a Hetzner server by name'
    s = hetz().servers.get_by_name(name)
    if s: s.delete()

In [ ]:
#| export
def hetz_keys() -> list:
    'List SSH keys registered in the Hetzner account as [{name, fingerprint}].'
    return L(hetz().ssh_keys.get_all()).map(lambda k: dict(name=k.name, fingerprint=k.fingerprint))

def hetz_key_names() -> list:
    'Return SSH key name strings for use in provision(ssh_keys=[...]).'
    return [k['name'] for k in hetz_keys()]

### Integration tests

Requires `HCLOUD_TOKEN` in the environment. Creates a minimal `cx11` server, verifies the lifecycle, then deletes it.

In [ ]:
#| eval: False
servers()

[{'name': 'vedicreader-cx32-hel', 'ip': '46.62.133.112', 'status': 'running'}]

In [ ]:
#| eval: False
_TOKEN = os.environ.get('HCLOUD_TOKEN')
_TEST_SERVER = 'fastops-test'

if not _TOKEN: print('HCLOUD_TOKEN not set — skipping hcloud integration tests')
else:
    # clean up any leftover from a previous run
    delete(_TEST_SERVER)

    # create — use cx23 (smallest current) with no cloud-init for a fast API test
    raw = create(_TEST_SERVER, server_type='cx23', location='hel1')
    ip = next(iter(raw))   # create() returns {ip: resp}
    assert ip, 'create() should return an IP'
    print(f'create OK: {ip}')

    # servers — new server appears in list
    svrs = servers()
    assert _TEST_SERVER in [s['name'] for s in svrs]
    print(f'servers() OK: {[s["name"] for s in svrs]}')

    # server_ip — matches what create() returned
    assert server_ip(_TEST_SERVER) == ip
    print(f'server_ip() OK: {ip}')

    # delete — server gone from list
    delete(_TEST_SERVER)
    assert _TEST_SERVER not in [s['name'] for s in servers()]
    print('delete() OK')
    print('All hcloud tests passed!')

create OK: 89.167.35.96
servers() OK: ['vedicreader-cx32-hel', 'fastops-test']
server_ip() OK: 89.167.35.96
delete() OK
All hcloud tests passed!


## SSH helpers

Pure subprocess-based SSH/rsync utilities — no paramiko dependency. `deploy()` syncs a Compose stack to a remote host and brings it up.

In [ ]:
#| export
def _ssh_base(host, user, key, port):
    a = ['ssh', '-o', 'StrictHostKeyChecking=accept-new']
    if key: a += ['-i', str(key)]
    if port != 22: a += ['-p', str(port)]
    return a + [f'{user}@{host}']

def run_ssh(host, *cmds, user='deploy', key=None, port=22, capture=False, check=True):
    'Run commands on remote host via SSH. capture=True returns stdout string.'
    r = subprocess.run(_ssh_base(host, user, key, port) + [' && '.join(cmds)],
                       capture_output=capture, text=capture, check=check)
    return r.stdout if capture else r

def sync(src, dst_path, host, user='deploy', key=None, port=22, exclude=None):
    'Rsync local path to remote host:dst_path. exclude: list of patterns for --exclude.'
    a = ['rsync', '-az', '--delete'] + (['-e', f'ssh -i {key}'] if key else [])
    for e in listify(exclude): a += ['--exclude', e]
    subprocess.run(a + [str(src), f'{user}@{host}:{dst_path}'], check=True)

def deploy(src, host, user='deploy', key=None, path='/srv/app', build=True, exclude=None):
    'Rsync local directory to remote host:path and run docker compose up -d.'
    run_ssh(host, f'mkdir -p {path}', user=user, key=key)
    sync(str(src).rstrip('/') + '/', path, host, user=user, key=key, exclude=exclude)
    cmd = f'cd {path} && docker compose up -d --remove-orphans' + (' --build' if build else '')
    return run_ssh(host, cmd, user=user, key=key)

## Verification helpers

Cloud-init runs asynchronously after `create()` returns. Use `wait_ssh()` to block until the server is reachable, then `check_cloud_init()` and `check_docker()` to confirm the bootstrap completed successfully before deploying.

In [ ]:
#| export
def wait_ssh(host, u='deploy', k=None, p=22, tout=120, interval=5):
    'Poll SSH until connection succeeds or raises TimeoutError.'
    dl = time.time() + tout
    while time.time() < dl:
        try: run_ssh(host, 'true', user=u, key=k, port=p); return True
        except Exception: time.sleep(interval)
    raise TimeoutError(f'SSH to {host} not ready after {tout}s')

def chk_cloud_init(host, u='deploy', k=None) -> str:
    'Return cloud-init status: done|running|error|unknown'
    try:
        o = run_ssh(host, 'sudo cloud-init status', user=u, key=k, capture=True)
        return o.split(': ', 1)[-1].strip() if ': ' in o else o.strip()
    except Exception: return 'unknown'

def chk_docker(host, u='deploy', k=None) -> bool:
    'Verify docker daemon is running and user can run containers.'
    try: run_ssh(host, 'docker info', user=u, key=k); return True
    except Exception: return False

## SSH key helpers

`load_pub_keys()` resolves a list of paths (or auto-detects `~/.ssh/id_*.pub`) into a flat list of public key strings ready for `vps_init()` and `multi_init()`.

In [ ]:
#| export
def load_pub_keys(paths=None) -> list:
    'Load SSH public key strings. paths=None → auto-detect from ~/.ssh/id_*.pub'
    paths = paths or list(Path.home().glob('.ssh/id_*.pub'))
    return [Path(p).read_text().strip() for p in listify(paths) if Path(p).exists()]

In [ ]:
keys = load_pub_keys()
print(f'Found {len(keys)} local SSH key(s)')

import tempfile as _tf
_f = Path(_tf.mktemp(suffix='.pub'))
_f.write_text('ssh-ed25519 TESTKEY comment')
assert load_pub_keys([str(_f)]) == ['ssh-ed25519 TESTKEY comment']
_f.unlink()
assert load_pub_keys(['/no_such_key.pub']) == []
print('load_pub_keys OK')

Found 4 local SSH key(s)
load_pub_keys OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()